### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model(model="gpt-5")
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high']}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002C4A1672510>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002C4A1672F9

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high']}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002C4A1672510>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions obj

In [4]:
model.invoke("Please provide the details of the movie Inception.")

AIMessage(content='Here are key details about the film Inception:\n\n- Title: Inception\n- Year: 2010\n- Genre: Science fiction heist thriller\n- Logline/Synopsis: Dom Cobb is a skilled thief who steals secrets by infiltrating people’s dreams. Offered a chance to erase his criminal record, he must achieve “inception”: planting an idea in the mind of heir Robert Fischer. Cobb assembles a team to execute a multilayered dream heist, but his own subconscious—haunted by the memory of his wife, Mal—threatens the mission and his return home.\n- Director: Christopher Nolan\n- Writer: Christopher Nolan\n- Producers: Emma Thomas, Christopher Nolan\n- Main cast:\n  - Leonardo DiCaprio as Dom Cobb\n  - Joseph Gordon-Levitt as Arthur\n  - Elliot Page (credited as Ellen Page) as Ariadne\n  - Tom Hardy as Eames\n  - Ken Watanabe as Saito\n  - Cillian Murphy as Robert Fischer\n  - Marion Cotillard as Mal\n  - Dileep Rao as Yusuf\n  - Tom Berenger as Browning\n  - Michael Caine as Miles\n- Music: Hans 

In [5]:
response = model_with_structure.invoke("Please provide the details of the movie Inception.")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [6]:
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    
model_with_structure=model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Please provide the details of the movie Inception.")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 674, 'prompt_tokens': 119, 'total_tokens': 793, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPReQTXJX7d0LUjn5N3BBkEIaVZMX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b469-58c3-7973-b412-206f904c8413-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 674, 'tot

### Nested Structure

In [7]:
class Actor(BaseModel):
    name:str=Field(description="The name of the actor")
    role:str=Field(description="The role of the actor in the movie")
    
class MovieDetails(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    actors:list[Actor]=Field(description="A list of actors in the movie")

In [8]:
model_with_structure=model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Please provide the details of the movie Inception.")
response

MovieDetails(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Tom Berenger', role='Peter Browning')])

### TypedDict

TypedDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]
    
model_withTypedDict=model.with_structured_output(MovieDict)

In [10]:
response = model_withTypedDict.invoke("Please provide the details of the movie Avengers.")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [11]:
model.profile

{'name': 'GPT-5',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high']}

### Data Classes

A data class is a class typically containing mainly data, although there aren't really any restrictions.

In [12]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str #The name of the person
    email: str #The email address of the person
    phone: str #The phone number of the person
    
agent = create_agent(model="gpt-5", response_format=ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')